# Audio Counterfactual Analysis via DiME

This notebook analyzes the results of the audio counterfactual experiments on the OpenMIC dataset.

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import PIL.Image
import pandas as pd
import librosa
import scienceplots

plt.style.use(['science', 'no-latex'])
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

REPORT_DIR = "../report/figures"
os.makedirs(REPORT_DIR, exist_ok=True)

## Figure 1: Qualitative Mel-Spectrogram Comparisons

Show side-by-side Input, Counterfactual, and Difference Map.

In [ ]:
def plot_spectrogram_comparison(exp_name, step=0, sample_idx=0):
    base_dir = f"../audio/results/{exp_name}/step_{step}"
    orig_path = os.path.join(base_dir, "original_spec", f"{sample_idx:06d}.png")
    cf_path = os.path.join(base_dir, "cf_spec", f"{sample_idx:06d}.png")
    diff_path = os.path.join(base_dir, "diff_spec", f"{sample_idx:06d}.png")
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, path, title in zip(axes, [orig_path, cf_path, diff_path], ["Original", "Counterfactual", "Difference"]):
        img = PIL.Image.open(path)
        ax.imshow(img, aspect="auto", origin="lower", cmap="magma")
        ax.set_title(title)
        ax.axis("off")
    
    plt.tight_layout()
    plt.savefig(os.path.join(REPORT_DIR, f"{exp_name}_step{step}_sample{sample_idx}_comparison.png"), dpi=300)
    plt.show()

# plot_spectrogram_comparison("exp1_guitar_remove", 0, 0)

## Figure 2: Diversity Assessment

Compare multiple counterfactuals for the same input (requires multiple runs/experiments).

In [ ]:
def plot_diversity(exp_names, step=0, sample_idx=0):
    n = len(exp_names)
    fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n+1), 4))
    
    # Original
    orig_path = f"../audio/results/{exp_names[0]}/step_{step}/original_spec/{sample_idx:06d}.png"
    axes[0].imshow(PIL.Image.open(orig_path), aspect="auto", origin="lower", cmap="magma")
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    for i, exp in enumerate(exp_names):
        cf_path = f"../audio/results/{exp}/step_{step}/cf_spec/{sample_idx:06d}.png"
        axes[i+1].imshow(PIL.Image.open(cf_path), aspect="auto", origin="lower", cmap="magma")
        axes[i+1].set_title(f"CF {i+1}")
        axes[i+1].axis("off")
        
    plt.tight_layout()
    plt.savefig(os.path.join(REPORT_DIR, f"diversity_sample{sample_idx}.png"), dpi=300)
    plt.show()

## Figure 3: Generation Pipeline

Visualize intermediate denoising steps (requires --save_intermediate True).

In [ ]:
def plot_pipeline(exp_name, step=0, sample_idx=0):
    inter_dir = f"../audio/results/{exp_name}/step_{step}/intermediate/{sample_idx:06d}"
    if not os.path.exists(inter_dir):
        print("Intermediate steps not found. Run with --save_intermediate True")
        return
    
    files = sorted(os.listdir(inter_dir))
    # Pick ~5 representative steps
    indices = np.linspace(0, len(files)-1, 5, dtype=int)
    
    fig, axes = plt.subplots(1, len(indices), figsize=(20, 4))
    for ax, idx in zip(axes, indices):
        img = PIL.Image.open(os.path.join(inter_dir, files[idx]))
        ax.imshow(img, aspect="auto", origin="lower", cmap="magma")
        ax.set_title(f"Iter {idx}")
        ax.axis("off")
        
    plt.tight_layout()
    plt.savefig(os.path.join(REPORT_DIR, f"pipeline_{exp_name}_sample{sample_idx}.png"), dpi=300)
    plt.show()

## Table 1: Main Results Table

In [ ]:
def compute_snr_db(x, y, eps=1e-8):
    noise = y - x
    p_signal = np.mean(x ** 2)
    p_noise = np.mean(noise ** 2)
    return 10 * np.log10((p_signal + eps) / (p_noise + eps))

def compute_lsd(x, y, sr=22050, n_fft=1024, hop_length=512, eps=1e-8):
    sx = np.abs(librosa.stft(x, n_fft=n_fft, hop_length=hop_length)) + eps
    sy = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length)) + eps
    lx = 20 * np.log10(sx)
    ly = 20 * np.log10(sy)
    return np.mean(np.sqrt(np.mean((lx - ly) ** 2, axis=0)))

def collect_all_metrics(exp_name, step=0):
    step_dir = f"../audio/results/{exp_name}/step_{step}"
    info_dir = os.path.join(step_dir, "info")
    
    # 1. Basic Stats
    flipped, l1s, mnac = [], [], []
    for f in os.listdir(info_dir):
        with open(os.path.join(info_dir, f), 'r') as j:
            d = json.load(j)
            flipped.append(d['flipped'])
            l1s.append(d['l1'])
            pb, pa = np.array(d['all_probs_before']), np.array(d['all_probs_after'])
            mask = np.ones_like(pb, dtype=bool)
            mask[d['target_class']] = False
            mnac.append(((pb[mask] > 0.5) != (pa[mask] > 0.5)).sum())
            
    # 2. Audio Metrics
    snrs, lsds = [], []
    orig_wav = os.path.join(step_dir, "original_wav")
    cf_wav = os.path.join(step_dir, "cf_wav")
    for f in sorted(os.listdir(orig_wav)):
        x, _ = librosa.load(os.path.join(orig_wav, f), sr=None)
        y, _ = librosa.load(os.path.join(cf_wav, f), sr=None)
        snrs.append(compute_snr_db(x, y))
        lsds.append(compute_lsd(x, y))
        
    # 3. Aggregated (Simplified placeholders for FAD/Similarity run manually)
    return {
        "Flip Ratio (%)": 100 * np.mean(flipped),
        "L1 Prox": np.mean(l1s),
        "MNAC": np.mean(mnac),
        "SNR (dB)": np.mean(snrs),
        "LSD": np.mean(lsds)
    }

all_exps = ["exp1_guitar_remove"]
df = pd.DataFrame([collect_all_metrics(e) for e in all_exps], index=all_exps)
df.to_latex(os.path.join(REPORT_DIR, "main_results.tex"))
display(df)